In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

In [30]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


In [11]:
import json
import pandas as pd
from datetime import datetime

# Load JSON file
with open('../NBAPropFinder/DATA/CSV_FILES/prizepicks_projections.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Create player name lookup from included array
player_names = {}
for elem in data.get('included', []):
    if elem.get('type') == 'new_player':
        player_names[elem['id']] = elem['attributes']['name']

# Extract projections into list
projections = []
for proj in data.get('data', []):
    if proj.get('type') == 'projection':
        player_id = proj['relationships']['new_player']['data']['id']
        player_name = player_names.get(player_id, 'Unknown')
        
        # Parse start_time and extract date
        start_time = proj['attributes']['start_time']
        game_date = datetime.fromisoformat(start_time).date()
        
        projections.append({
            'NAME': player_name,
            'CATEGORY': proj['attributes']['stat_type'],
            'LINE': proj['attributes']['line_score'],
            'ODDS TYPE': proj['attributes']['odds_type'],
            'COMMENCE TIME': game_date
        })

# Create DataFrame
df = pd.DataFrame(projections)
df = df[(df['CATEGORY'] == 'Points') & (df['ODDS TYPE'] == 'goblin')]
df

,NAME,CATEGORY,LINE,ODDS TYPE,COMMENCE TIME
100,Donovan Mitchell,Points,25.5,goblin,2025-11-19
103,Donovan Mitchell,Points,24.5,goblin,2025-11-19
104,Kevin Durant,Points,24.5,goblin,2025-11-19
136,Kevin Durant,Points,22.5,goblin,2025-11-19
143,Kevin Durant,Points,21.5,goblin,2025-11-19
...,...,...,...,...,...
6128,Kris Murray,Points,6.5,goblin,2025-11-19
6155,Jalen Smith,Points,5.5,goblin,2025-11-19
6156,Kris Murray,Points,5.5,goblin,2025-11-19
6171,Isaac Okoro,Points,4.5,goblin,2025-11-19


In [17]:
def calculate2LegBets(data, bookmakers, model, features, current_date, edge_threshold=0.05, top_n=10, 
                 variance_inflation=1.1, 
                 use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, stake=100,
                 enforce_downside_skew: bool = False, skew_override: float | None = None,
                 max_player_appearances: int = 3):

    # Set random seed once before the loop for reproducibility
    np.random.seed(42)
    rng = np.random.RandomState(42)
    
    # Store player metadata (same for all lines of a player)
    player_metadata = {}
    
    # Get all unique players
    available_players = bookmakers['NAME'].unique()
    if len(available_players) < 2:
        print("Not enough players for 2-leg bets")
        return pd.DataFrame()
    
    print(f"Pre-computing predictions for {len(available_players)} players...")
    current_date_str = pd.to_datetime(current_date).strftime('%Y-%m-%d')
    
    # Pre-compute predictions and metadata for each player (once per player, not per line)
    for player in available_players:
        # Map player name
        mapped_player = nameDict.get(player, player)
        
        # Get prediction data
        pred_data = get_cached_prediction(mapped_player, data, model, features, current_date, 
                                         projectedStartingFive, mainStartingFive, teamStarPlayer)
        if pred_data is None:
            continue
        
        # Get player data for team lookup
        player_data = data[data['PLAYER_NAME'] == mapped_player]
        if player_data.empty:
            continue
        
        # Get team
        player_team = player_data['TEAM_ABBREVIATION'].iloc[-1]
        
        # Get opponent using findOpp
        opp_team, _ = findOpp(mapped_player, player_data, current_date_str)
        if opp_team is None:
            continue
        
        # Store metadata for this player (will be reused for all their lines)
        player_metadata[player] = {
            'mapped_name': mapped_player,
            'prediction': pred_data,
            'team': player_team,
            'opponent': opp_team
        }
    
    # Filter to only players with valid metadata
    available_players = [p for p in available_players if p in player_metadata]
    
    if len(available_players) < 2:
        print("Not enough players with valid predictions for 2-leg bets")
        return pd.DataFrame()
    
    # Create list of available legs (player-line-odds_type combinations)
    # Each leg represents a unique player+line+odds_type combination
    available_legs = []
    odds_type_col = 'ODDS TYPE' if 'ODDS TYPE' in bookmakers.columns else ('odds_type' if 'odds_type' in bookmakers.columns else None)
    
    for player in available_players:
        # Get all betting lines for this player
        player_bets = bookmakers[bookmakers['NAME'] == player]
        
        if player_bets.empty:
            continue
        
        # Group by unique (LINE, ODDS TYPE) combinations to avoid duplicates
        if odds_type_col:
            # Create unique combinations of line and odds_type
            for _, bet_row in player_bets.drop_duplicates(subset=['LINE', odds_type_col]).iterrows():
                line = float(bet_row['LINE'])
                odds_type = str(bet_row[odds_type_col]).strip().lower() if pd.notna(bet_row[odds_type_col]) else 'standard'
                
                available_legs.append({
                    'player': player,
                    'line': line,
                    'odds_type': odds_type,
                    'bet_row': bet_row
                })
        else:
            # No odds_type column, just use unique lines
            for _, bet_row in player_bets.drop_duplicates(subset=['LINE']).iterrows():
                line = float(bet_row['LINE'])
                
                available_legs.append({
                    'player': player,
                    'line': line,
                    'odds_type': 'standard',
                    'bet_row': bet_row
                })
    
    if len(available_legs) < 2:
        print("Not enough legs for 2-leg combinations")
        return pd.DataFrame()
    
    print(f"Found {len(available_legs)} available legs from {len(available_players)} players")
    
    # Generate valid 2-leg combinations (prevent same team)
    valid_combinations = []
    for leg1, leg2 in combinations(available_legs, 2):
        p1 = leg1['player']
        p2 = leg2['player']
        team1 = player_metadata[p1]['team']
        team2 = player_metadata[p2]['team']
        
        # Prevent same-team combinations
        if team1 == team2:
            continue
        
        valid_combinations.append((leg1, leg2))
    
    print(f"Generated {len(valid_combinations)} valid 2-leg combinations")
    
    # Pre-compute market probability (constant for all)
    market_prob = impliedProb(-137)
    market_prob_combined = market_prob ** 2
    
    # Pre-compute constants
    corr_factor = 0.90
    
    # Helper function for sigma flag
    def flag_sigma(s):
        if s <= 5.0:
            return 'Low'
        elif s <= 6.0:
            return 'Med'
        else:
            return 'High'
    
    # Import skewnorm once
    from scipy.stats import skewnorm
    
    results = []
    batch_size = n_simulations * 2

    for leg1, leg2 in valid_combinations:
        # Get player info
        p1 = leg1['player']
        p2 = leg2['player']
        mapped_p1 = player_metadata[p1]['mapped_name']
        mapped_p2 = player_metadata[p2]['mapped_name']
        
        # Get pre-computed prediction data
        pred1_data = player_metadata[p1]['prediction']
        pred2_data = player_metadata[p2]['prediction']
        
        pred1_val = pred1_data['prediction']
        pred2_val = pred2_data['prediction']
        sigma1 = pred1_data['sigma'] * variance_inflation
        sigma2 = pred2_data['sigma'] * variance_inflation
        skew1 = pred1_data['skew']
        skew2 = pred2_data['skew']
        
        # Get lines and odds types
        line1 = leg1['line']
        line2 = leg2['line']
        odds_type1 = leg1['odds_type']
        odds_type2 = leg2['odds_type']
        
        # Calculate payout_multiple based on odds type combination
        if odds_type1 == 'demon' and odds_type2 == 'demon':
            # 2 demon: 6x
            payout_multiple = 6.0
        elif odds_type1 == 'goblin' and odds_type2 == 'goblin':
            # 2 goblin: 2x
            payout_multiple = 2.0
        elif (odds_type1 == 'demon' and odds_type2 == 'goblin') or (odds_type1 == 'goblin' and odds_type2 == 'demon'):
            # demon + goblin: 3.5x
            payout_multiple = 3.5
        elif (odds_type1 == 'demon' and odds_type2 == 'standard') or (odds_type1 == 'standard' and odds_type2 == 'demon'):
            # demon + standard: 3.5x
            payout_multiple = 3.5
        elif (odds_type1 == 'goblin' and odds_type2 == 'standard') or (odds_type1 == 'standard' and odds_type2 == 'goblin'):
            # goblin + standard: 3x
            payout_multiple = 3.0
        elif odds_type1 == 'standard' and odds_type2 == 'standard':
            # 2 standard: 3x
            payout_multiple = 3.0
        else:
            # Default fallback (shouldn't happen, but safety)
            payout_multiple = 3.0
        
        b = payout_multiple - 1.0
        
        # Distribution parameters
        mu1 = pred1_val
        mu2 = pred2_val
        
        # Determine skew parameters
        a1 = skew_override if skew_override is not None else (-2.0 if enforce_downside_skew else skew1)
        a2 = skew_override if skew_override is not None else (-2.0 if enforce_downside_skew else skew2)
        
        # Confidence intervals and sigma flags
        ci1 = (max(0, mu1 - 1.96 * sigma1), mu1 + 1.96 * sigma1)
        ci2 = (max(0, mu2 - 1.96 * sigma2), mu2 + 1.96 * sigma2)
        width1 = round(ci1[1] - ci1[0], 2)
        width2 = round(ci2[1] - ci2[0], 2)
        sigma_flag1 = flag_sigma(sigma1)
        sigma_flag2 = flag_sigma(sigma2)
        
        # Calculate probabilities using Monte Carlo or analytical method
        if use_monte_carlo:
            # Player 1
            sim1_list = []
            while len(sim1_list) < n_simulations:
                batch = skewnorm.rvs(a1, loc=mu1, scale=sigma1, size=batch_size, random_state=rng)
                positive_batch = batch[batch >= 0]
                sim1_list.extend(positive_batch)
                if len(sim1_list) >= n_simulations:
                    break
            sim1 = np.array(sim1_list[:n_simulations])
            
            # Player 2
            sim2_list = []
            while len(sim2_list) < n_simulations:
                batch = skewnorm.rvs(a2, loc=mu2, scale=sigma2, size=batch_size, random_state=rng)
                positive_batch = batch[batch >= 0]
                sim2_list.extend(positive_batch)
                if len(sim2_list) >= n_simulations:
                    break
            sim2 = np.array(sim2_list[:n_simulations])
            
            p1_over_raw = np.mean(sim1 > line1)
            p2_over_raw = np.mean(sim2 > line2)
        else:
            # Analytical method using conditional probability
            # Player 1
            p1_above_line = 1 - skewnorm.cdf(line1, a1, loc=mu1, scale=sigma1)
            p1_non_negative = 1 - skewnorm.cdf(0, a1, loc=mu1, scale=sigma1)
            p1_over_raw = (p1_above_line / p1_non_negative) if p1_non_negative > 1e-10 else max(0.0, min(1.0, p1_above_line))
            
            # Player 2
            p2_above_line = 1 - skewnorm.cdf(line2, a2, loc=mu2, scale=sigma2)
            p2_non_negative = 1 - skewnorm.cdf(0, a2, loc=mu2, scale=sigma2)
            p2_over_raw = (p2_above_line / p2_non_negative) if p2_non_negative > 1e-10 else max(0.0, min(1.0, p2_above_line))
        
        # Determine model sides based on predictions vs lines AND odds type
        # For goblin/demon: force over; for standard: use prediction vs line logic
        if odds_type1 in ['goblin', 'demon']:
            # Goblins/Demons only allow over bets
            model_side1 = 'over'
            p1 = p1_over_raw
            p1_raw = p1_over_raw
        elif pred1_val > line1:
            # Standard: prediction above line -> bet over
            model_side1 = 'over'
            p1 = p1_over_raw
            p1_raw = p1_over_raw
        else:
            # Standard: prediction below line -> bet under
            model_side1 = 'under'
            p1 = 1 - p1_over_raw
            p1_raw = 1 - p1_over_raw
            
        if odds_type2 in ['goblin', 'demon']:
            # Goblins/Demons only allow over bets
            model_side2 = 'over'
            p2 = p2_over_raw
            p2_raw = p2_over_raw
        elif pred2_val > line2:
            # Standard: prediction above line -> bet over
            model_side2 = 'over'
            p2 = p2_over_raw
            p2_raw = p2_over_raw
        else:
            # Standard: prediction below line -> bet under
            model_side2 = 'under'
            p2 = 1 - p2_over_raw
            p2_raw = 1 - p2_over_raw
        
        # Calculate combined probability and EV with correlation adjustment (using raw probabilities)
        p_both_raw = p1_raw * p2_raw
        p_both = p_both_raw * corr_factor
        ev = payout_multiple * p_both - 1
        ev_dollars = ev * stake
        
        # Kelly criterion with variance-adjusted constraint
        kelly_full = max(0.0, (b * p_both - (1 - p_both)) / b) if b > 0 else 0.0
        
        # Edge calculation (using raw probabilities for accuracy)
        edge1 = p1_raw - market_prob
        edge2 = p2_raw - market_prob
        
        # Calculate combined probabilities and edge (using raw probabilities)
        combined_model_prob = p1_raw * p2_raw
        combined_edge = combined_model_prob - market_prob_combined
        
        # Recommendation based on multiple criteria
        recommendation = 1 if (abs(line1 - pred1_val) > 4.5 and abs(line2 - pred2_val) > 4.5) else 0
        
        results.append({
            'NAME 1': mapped_p1,
            'NAME 2': mapped_p2,
            'LINE 1': line1,
            'LINE 2': line2,
            'PREDICTION 1': round(pred1_val, 2),
            'PREDICTION 2': round(pred2_val, 2),
            'MODEL SIDE 1': model_side1,
            'MODEL SIDE 2': model_side2,
            'ODDS TYPE 1': odds_type1,
            'ODDS TYPE 2': odds_type2,
            'PAYOUT MULTIPLE': payout_multiple,  # NEW: Include payout multiple in output
            'PROB 1': round(p1, 3),
            'PROB 2': round(p2, 3),
            'PROB BOTH': round(p_both, 4),
            'EDGE 1': round(edge1, 3),
            'EDGE 2': round(edge2, 3),
            'COMBINED EDGE': round(combined_edge, 3),
            'EV$': round(ev_dollars, 2),
            'KELLY FULL': round(kelly_full, 3),
            'RECOMMENDATION': recommendation,
            'CONFIDENCE INTERVAL 1': f"({ci1[0]:.1f}, {ci1[1]:.1f})",
            'CONFIDENCE INTERVAL 2': f"({ci2[0]:.1f}, {ci2[1]:.1f})",
            'INTERVAL WIDTH 1': width1,
            'INTERVAL WIDTH 2': width2,
            'SIGMA 1': round(sigma1, 2),
            'SIGMA 2': round(sigma2, 2),
            'SIGMA FLAG 1': sigma_flag1,
            'SIGMA FLAG 2': sigma_flag2,
            'EXPECTED ROI': round((ev_dollars / stake) * 100, 1),
            'SIMULATION METHOD': 'Monte Carlo' if use_monte_carlo else 'Analytical'
        })
    
    results_df = pd.DataFrame(results)
    
    # Apply player frequency limit for diversification
    if max_player_appearances is not None and len(results_df) > 0:
        # Sort by EV descending to prioritize best bets
        results_df = results_df.sort_values('EV$', ascending=False).reset_index(drop=True)
        
        # Track how many times each player appears
        player_count = defaultdict(int)
        selected_rows = []
        
        for idx, row in results_df.iterrows():
            p1 = row['NAME 1']
            p2 = row['NAME 2']
            
            # Check if adding this combination would exceed the limit for any player
            if (player_count[p1] < max_player_appearances and 
                player_count[p2] < max_player_appearances):
                selected_rows.append(idx)
                player_count[p1] += 1
                player_count[p2] += 1
        
        results_df = results_df.loc[selected_rows].reset_index(drop=True)
        print(f"Applied player frequency limit ({max_player_appearances} max appearances per player)")
        print(f"Selected {len(selected_rows)} combinations from {len(results)} candidates")
    
    return results_df

In [18]:
df = calculate2LegBets(s26, df, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=None)

KeyError: 'NAME'

In [14]:
df.sort_values('EV$', ascending=False).head(10)

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,ODDS TYPE 1,ODDS TYPE 2,PAYOUT MULTIPLE,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,INTERVAL WIDTH 1,INTERVAL WIDTH 2,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,EXPECTED ROI,SIMULATION METHOD
9567,Bennedict Mathurin,Zion Williamson,14.5,14.5,22.81,22.53,over,over,goblin,goblin,2.0,0.959,0.916,0.7904,0.381,0.338,0.544,5.81,0.581,1,"(13.4, 32.2)","(11.1, 34.0)",18.73,22.88,4.78,5.84,Low,Med,58.1,Analytical
9656,Bennedict Mathurin,Josh Giddey,14.5,14.5,22.81,25.27,over,over,goblin,goblin,2.0,0.959,0.911,0.7866,0.381,0.333,0.540,5.73,0.573,1,"(13.4, 32.2)","(9.6, 41.0)",18.73,31.42,4.78,8.02,Low,High,57.3,Analytical
9513,Bennedict Mathurin,Pelle Larsson,14.5,5.5,22.81,13.30,over,over,goblin,goblin,2.0,0.959,0.898,0.7748,0.381,0.320,0.527,5.50,0.550,1,"(13.4, 32.2)","(0.1, 26.5)",18.73,26.49,4.78,6.76,Low,High,55.0,Analytical
5147,Reed Sheppard,Bennedict Mathurin,7.5,14.5,14.27,22.81,over,over,goblin,goblin,2.0,0.894,0.959,0.7715,0.316,0.381,0.523,5.43,0.543,1,"(3.4, 25.1)","(13.4, 32.2)",21.68,18.73,5.53,4.78,Med,Low,54.3,Analytical
9510,Bennedict Mathurin,Davion Mitchell,14.5,5.5,22.81,12.40,over,over,goblin,goblin,2.0,0.959,0.882,0.7616,0.381,0.304,0.512,5.23,0.523,1,"(13.4, 32.2)","(0.0, 25.0)",18.73,24.99,4.78,6.42,Low,High,52.3,Analytical
9572,Bennedict Mathurin,Jeremiah Fears,14.5,9.5,22.81,18.00,over,over,goblin,goblin,2.0,0.959,0.881,0.7608,0.381,0.303,0.511,5.22,0.522,1,"(13.4, 32.2)","(3.5, 32.5)",18.73,28.98,4.78,7.39,Low,High,52.2,Analytical
9670,Bennedict Mathurin,Ayo Dosunmu,14.5,7.5,22.81,14.95,over,over,goblin,goblin,2.0,0.959,0.880,0.7599,0.381,0.302,0.510,5.20,0.520,1,"(13.4, 32.2)","(2.0, 27.9)",18.73,25.91,4.78,6.61,Low,High,52.0,Analytical
9602,Bennedict Mathurin,Kyshawn George,14.5,9.5,22.81,17.97,over,over,goblin,goblin,2.0,0.959,0.875,0.7553,0.381,0.297,0.505,5.11,0.511,1,"(13.4, 32.2)","(3.0, 32.9)",18.73,29.85,4.78,7.62,Low,High,51.1,Analytical
9627,Bennedict Mathurin,Rob Dillingham,14.5,3.5,22.81,8.80,over,over,goblin,goblin,2.0,0.959,0.875,0.7548,0.381,0.297,0.505,5.10,0.510,1,"(13.4, 32.2)","(0.0, 20.4)",18.73,20.44,4.78,5.94,Low,Med,51.0,Analytical
9679,Bennedict Mathurin,Jalen Smith,14.5,5.5,22.81,12.19,over,over,goblin,goblin,2.0,0.959,0.872,0.7529,0.381,0.294,0.502,5.06,0.506,1,"(13.4, 32.2)","(0.0, 25.2)",18.73,25.18,4.78,6.63,Low,High,50.6,Analytical
